In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

print(os.listdir('/content/drive/MyDrive/DVA_Project/data/interim'))

['extracted.csv']


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

In [7]:
FILE_PATH = '/content/drive/MyDrive/DVA_Project/data/interim/extracted.csv'

df = pd.read_csv(FILE_PATH)

print(df.shape)
df.head()

(12575, 11)


,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [8]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

numeric_cols = ['price_per_unit', 'quantity', 'total_spent']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [9]:
cat_cols = ['category', 'payment_method', 'location']

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

In [10]:
df['discount_applied'] = (
    df['discount_applied']
    .astype(str)
    .str.strip()
    .str.upper()
    .map({'TRUE': True, 'FALSE': False})
)

df['discount_applied'] = df['discount_applied'].fillna(False)

/tmp/ipykernel_19591/1421791326.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['discount_applied'] = df['discount_applied'].fillna(False)


In [11]:
# total_spent missing
mask = df['total_spent'].isnull() & df['price_per_unit'].notnull() & df['quantity'].notnull()
df.loc[mask, 'total_spent'] = df['price_per_unit'] * df['quantity']

# price_per_unit missing
mask = df['price_per_unit'].isnull() & df['total_spent'].notnull() & df['quantity'].notnull()
df.loc[mask, 'price_per_unit'] = df['total_spent'] / df['quantity']

# quantity missing
mask = df['quantity'].isnull() & df['price_per_unit'].notnull() & df['total_spent'].notnull()
df.loc[mask, 'quantity'] = df['total_spent'] / df['price_per_unit']

In [12]:
df = df.dropna(subset=['price_per_unit', 'quantity', 'total_spent'])

In [13]:
df['item'] = df['item'].fillna('Unknown')

In [14]:
df = df[(df['price_per_unit'] > 0) & (df['quantity'] > 0) & (df['total_spent'] > 0)]

In [15]:
diff = df['price_per_unit'] * df['quantity'] - df['total_spent']
print("Inconsistent rows:", (abs(diff) > 1e-6).sum())

Inconsistent rows: 0


In [16]:
print(df.shape)
print("\nMissing:\n", df.isnull().sum())

(11971, 11)

Missing:
 transaction_id      0
customer_id         0
category            0
item                0
price_per_unit      0
quantity            0
total_spent         0
payment_method      0
location            0
transaction_date    0
discount_applied    0
dtype: int64


In [17]:
df.to_csv('/content/drive/MyDrive/DVA_Project/data/interim/cleaned.csv', index=False)